In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import pylab as pl
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df = pd.read_csv("ahs_dataset_nisr.csv")
df

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df['recommended_crop'].value_counts()

In [ ]:
sns.kdeplot(data=df['soil_ph'], shade=True)

In [ ]:
sns.kdeplot(data=df['nitrogen'], shade=True)

In [ ]:
sns.kdeplot(data=df['phosphorus'], shade=True)

In [ ]:
sns.kdeplot(data=df['potassium'], shade=True)

In [ ]:
sns.kdeplot(data=df['avg_temperature_C'], shade=True)

In [ ]:
sns.kdeplot(data=df['avg_humidity_pct'], shade=True)

In [ ]:
sns.kdeplot(data=df['annual_rainfall_mm'], shade=True)

## Feature Engineering & Data Preparation
Encode categorical columns (district, province), fill missing yield values with 0, and define the full feature set used for model training (as per Section 3.3 of the proposal).

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode district and province
le_district = LabelEncoder()
le_province  = LabelEncoder()
df['district_enc'] = le_district.fit_transform(df['district_real'])
df['province_enc']  = le_province.fit_transform(df['province'])

# Fill missing yield values with 0
yield_columns = [
    'yield_maize_t_ha', 'yield_beans_t_ha', 'yield_cassava_t_ha',
    'yield_rice_t_ha',  'yield_irish_potato_t_ha', 'yield_tomato_t_ha',
    'yield_sweet_potato_t_ha', 'yield_banana_t_ha'
]
for col in yield_columns:
    df[col] = df[col].fillna(0)

print("Encoding complete. Missing values after fill:")
print(df[yield_columns].isnull().sum())

In [ ]:
# 8 core agro-environmental features (Section 3.3) + location + farming practices + yields
features = [
    'soil_ph', 'nitrogen', 'phosphorus', 'potassium',
    'annual_rainfall_mm', 'avg_temperature_C', 'avg_humidity_pct', 'altitude_m',
    'district_enc', 'province_enc', 'farm_size_ha', 'soc',
    'used_improved_seed', 'used_inorganic_fert', 'used_organic_fert',
    'used_pesticide', 'used_irrigation', 'num_crops_grown',
    'yield_maize_t_ha', 'yield_beans_t_ha', 'yield_cassava_t_ha',
    'yield_rice_t_ha',  'yield_irish_potato_t_ha', 'yield_tomato_t_ha',
    'yield_sweet_potato_t_ha', 'yield_banana_t_ha'
]

X = df[features]
y = df['recommended_crop']

# Encode target labels
le_crop = LabelEncoder()
y_encoded = le_crop.fit_transform(y)

print("Feature matrix shape:", X.shape)
print("Target classes:", list(le_crop.classes_))

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train / 30% test as per Section 3.4
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Test set size     : {X_test.shape[0]} samples")

## Model Training & Comparison
Four supervised classification algorithms are trained and compared as specified in Section 3.4: **Random Forest**, **Decision Tree**, **Naive Bayes**, and **K-Nearest Neighbor (KNN)**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

models = {
    'Random Forest' : RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree' : DecisionTreeClassifier(random_state=42),
    'Naive Bayes'   : GaussianNB(),
    'KNN'           : KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        'Accuracy'  : round(accuracy_score(y_test, pred),  4),
        'Precision' : round(precision_score(y_test, pred, average='weighted'), 4),
        'Recall'    : round(recall_score(y_test, pred,    average='weighted'), 4),
        'F1-Score'  : round(f1_score(y_test, pred,        average='weighted'), 4),
    }

results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
results_df.plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='black')
plt.title('Model Comparison — Accuracy, Precision, Recall, F1-Score', fontsize=13)
plt.xlabel('Model')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model based on F1-Score (as per Section 3.4)
best_name  = results_df['F1-Score'].idxmax()
best_model = models[best_name]

print(f"Best model selected : {best_name}")
print(f"F1-Score            : {results_df.loc[best_name, 'F1-Score']}")

In [ ]:
from sklearn.metrics import classification_report

best_pred = best_model.predict(X_test)
print(f"Classification Report — {best_name}\n")
print(classification_report(y_test, best_pred, target_names=le_crop.classes_))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_crop.classes_)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title(f'Confusion Matrix — {best_name}', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(11, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importances — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Save best model
joblib.dump(best_model,  'train_model.joblib')

# Save encoders for use in the Django backend
joblib.dump(le_crop,     'label_encoder.joblib')
joblib.dump(le_district, 'district_encoder.joblib')
joblib.dump(le_province, 'province_encoder.joblib')

print(f"Model saved  : train_model.joblib  ({best_name})")
print("Encoders saved : label_encoder.joblib, district_encoder.joblib, province_encoder.joblib")

In [ ]:
# Load model and make a sample prediction
loaded_model = joblib.load('train_model.joblib')
loaded_le    = joblib.load('label_encoder.joblib')

# Example: a household in Southern province with known soil & climate values
sample = pd.DataFrame([[
    6.1,   # soil_ph
    0.35,  # nitrogen
    0.12,  # phosphorus
    0.36,  # potassium
    900.0, # annual_rainfall_mm
    22.0,  # avg_temperature_C
    75.0,  # avg_humidity_pct
    1500,  # altitude_m
    0,     # district_enc  (encoded district index)
    3,     # province_enc  (encoded province index)
    1.2,   # farm_size_ha
    2.5,   # soc
    1,     # used_improved_seed
    1,     # used_inorganic_fert
    0,     # used_organic_fert
    0,     # used_pesticide
    0,     # used_irrigation
    2,     # num_crops_grown
    0.0,   # yield_maize_t_ha
    0.0,   # yield_beans_t_ha
    1.8,   # yield_cassava_t_ha
    0.0,   # yield_rice_t_ha
    0.0,   # yield_irish_potato_t_ha
    0.0,   # yield_tomato_t_ha
    0.0,   # yield_sweet_potato_t_ha
    2.1,   # yield_banana_t_ha
]], columns=features)

pred_encoded = loaded_model.predict(sample)
pred_crop    = loaded_le.inverse_transform(pred_encoded)
print("Recommended crop:", pred_crop[0])